# Knowledge-Based Robotic Decision-Making
Welcome to the third Chapter of our hands-on course!
Today, you will focus on understanding how a knowledge base supports robotic decision-making. You’ll learn to query the knowledge base to identify necessary actions for tasks, like how to perceive the milk inside the fridge.

## Why is this important?
Imagine the following RoboCup@Home Challenge-Use Case: The GPSR (General Purpose Service Robot) challenge, in which the robot can be asked to do basically anything within a household environment. For example, tasks like "bring me the cup from the living room table", "hand me the cup from the coffee table" or "please bring me a cup from the table in the living room" can all mean the same objects. KnowRob can make it easier for us to adress this challenge, by allowing us to refer to the sme object in multiple ways.

### Goal
By the end of the session, you will have successfully made queries to the knowledge base, enabling the robot to determine the steps required to complete its tasks.


## Step 1: Initialization

Import the necessary modules and define the objects in the environment, as we did in Day 2.

In [ ]:
from pycram.ros.tf_broadcaster import TFBroadcaster
from pycram.ros.viz_marker_publisher import VizMarkerPublisher
from pycram.worlds.bullet_world import BulletWorld
from pycram.designators.action_designator import *
from pycram.designators.location_designator import *
from pycram.designators.object_designator import *
from pycram.datastructures.enums import ObjectType, WorldMode, TorsoState
from pycram.datastructures.pose import Pose
from pycram.process_module import simulated_robot
from pycram.object_descriptors.urdf import ObjectDescription
from pycram.world_concepts.world_object import Object
from pycram.datastructures.dataclasses import Color
from pycram.external_interfaces.rosprolog_interface import Prolog as knowrob_client
import pycram.ros.joint_state_subscriber as joint_state


extension = ObjectDescription.get_file_extension()

world = BulletWorld(WorldMode.DIRECT)
world.allow_publish_debug_poses = True
viz = VizMarkerPublisher(interval=0.25)
tf = TFBroadcaster()

robot_name = "pr2"
robot = Object(robot_name, ObjectType.ROBOT, f"{robot_name}{extension}", pose=Pose([1, 2, 0]))

apartment = Object("apartment", ObjectType.ENVIRONMENT, f"apartment-small{extension}")
milk = Object("milk", ObjectType.MILK, "milk.stl", pose=Pose([0.5, 2.5, 1], [0, 0, 0, 1]))
milk.color = Color(0, 0, 1, 1)
milk_desig = BelieveObject(names=["milk"])
robot_desig = BelieveObject(names=[robot_name])
apartment_desig = BelieveObject(names=["apartment"])
joint_state_subscriber = joint_state.JointAngleReader()
print("Ready for the next cell.")


### Step 2: Learn how to Query the Knowledge Base

Before we can start quering our knowledge base, we have to establish a connection to it by creating an object for the client:

In [ ]:
knowrob = knowrob_client()
print("Ready for the next cell.")

If the execution was successfull, (you should see something like "[KnowRbo] done"), we can test the connection by sending a test query written in Prolog:
**Important** note: KnowRob uses Prolog as a query language, which allows it to be very powerful, but takes a while to get used to. Think of it as trying to express what you would like to know in a logic formula, and prolog will try and find a solution which matches your query.

In [ ]:
knowrob.once("member(X, [1,2,3]).")

Within the PyCram knowrob interface, we differenciate between wanting to receive just one result, or all possible solutions. For this, there are essentially two query functions: **once** and **all_solutions**. In most instances, it is fine to just use **once**. However **all_solutions** might be useful, for building failure handling functionality, in order to iterate over all the possible solutions. To see the difference for the member function, try it out with **all_solutions** instead of **once**. 

<span style="color:red">Question:</span>: Do you notice any other difference between the two functions?

In [ ]:
#Add your code here

<details>

<summary>Solution</summary>

```python
knowrob.all_solutions('member(X,[1,2,3]).')
```

The result of **once** is a dictionary, while the result of **all_solutions** is a list of dictionaries. Keep this in mind when working on the results of KnowRob. 
</details>

Let's briefly discuss what this query does. The **member** predicate is True, if X is a member of the given List. Since we have not specified a value for X, Prolog will list us all possible solutions for X, which in this case are all elements of the provided list. However, if we replace X with a value which is in or not in the list, we get True/False as a result respectively.

<span style="color:red">Plyground</span>: Try to find out if the number 1 is a member of the list [1,2,3], and then check if the number 4 is a member of the list [1,2,3].


In [ ]:
#Add your code here

<details>

<summary> Solution </summary>

```python
knowrob.once("member(1, [1,2,3]).")
knowrob.all_solutions("member(1, [1,2,3]).")

knowrob.once("member(4, [1,2,3]).")
knowrob.all_solutions("member(4, [1,2,3]).")
```
</details>

<details> 

<summary> Solution & Result</summary>

```python
In [10]: knowrob.once("member(1, [1,2,3]).")
Out[10]: True

In [11]: knowrob.all_solutions("member(1, [1,2,3]).")
Out[11]: True

In [12]: knowrob.once("member(4, [1,2,3]).")
Out[12]: False

In [13]: knowrob.all_solutions("member(4, [1,2,3]).")
Out[13]: False
```

</details>

<b> <span style="color:blue">Prolog in a Nutshell:</span>: </b>
The most important thing to know about Prolog is, that everything starting with a capital letter is a variabe which Prolog will fill with knowledge which matches the query, and that you can chain queries with a comma, since that is a logical "and". You also have to end your queries with a dot ".". Below you can find a cheat sheet for Prolog syntax. We won't need most of these things unless you really want to get in-depth into Prolog coding, but in case you are curious, it is a brief reference.

<details>

<summary><b> Prolog cheat sheet </b> </summary>


| Symbol          | Meaning                                | Example                                               |
|-----------------|----------------------------------------|-------------------------------------------------------|
| `.`             | End of clause or query                 | `likes(john, pizza).`                                 |
| `,`             | Logical AND                            | `happy(X), healthy(X).`                               |
| `;`             | Logical OR                             | `happy(X); sad(X).`                                   |
| `:-`            | "If" (defines a rule)                  | `happy(X) :- enjoys(X, Y), positive(Y).`              |
| `?-`            | Start a query                          | `?- likes(john, pizza).`                              |
| `=`             | Unification (binds values)             | `X = john.`                                           |
| `is`            | Arithmetic assignment                  | `X is 3 + 4.`                                         |
| `<`, `>`, `=<`, `>=` | Comparison operators        | `X > 5, Y =< 10.`                                     |
| `\=`            | Not equal                              | `X \= john.`                                          |
| `[]`            | Empty list                             | `X = [].`                                             |
| `\|`            | List cons (head and tail separator)    | `[H \| T] = [1, 2, 3].`                               |
| `_`             | Anonymous variable (ignored)           | `likes(_, pizza).`                                    |
| `\+`            | Negation                               | `\+ happy(X).`                                        |
| `!`             | Cut (prevents backtracking)            | `happy(X) :- enjoys(X, Y), !, positive(Y).`           |


If you are curious about Prolog as a language, you can find some Prolog tutorials here: <url> https://www.swi-prolog.org/pldoc/man?section=quickstart </url>

</details>

### Step 3: Learn why we need a Knowledge Base
Now that you know your way around Prolog, we would like to go back to our scenario of getting the milk from the fridge. Since the robot doesn't yet know where the milk is located, we could see if knowrob does, by posting a query. Let's develop this query step by step.

First, let's check if any object of Class Milk exists:

In [ ]:
knowrob.all_solutions("instance_of(Instance, suturo:'Milk').")

<details>

<summary> Result </summary>

```python
[{'Instance': 'http://www.ease-crc.org/ont/SUTURO.owl#Milk_REOCKXLU'}]
```

</details>

With the **instance_of** query, we can check if an instance of a Class exists. **Instance** in this case is the Variable which gets filled with the Object url/iri + ID. The ID of the object can change, since the object is created whenever KnowRob gets launched. So if you restart it, you will get a new Object ID. 
The **suturo** prefix, denotes that the **Milk** Class is defined in the Suturo ontology. 

The **instance_of** predicate, also works the other way around. So if you have an instance and would like to know which class it belongs to, you can do the following:

In [ ]:
knowrob.all_solutions("instance_of('http://www.ease-crc.org/ont/SUTURO.owl#Milk_REPLACE_WITH_ID_FROM_RESULT_ABOVE', Class).")

<details>

<summary> Result </summary>

```python
[{'Class': 'http://www.w3.org/2002/07/owl#NamedIndividual'},
 {'Class': 'http://www.ease-crc.org/ont/SUTURO.owl#Milk'}]

```

</details>

In this case we get two results since the Milk-Object is defined as an Object of the Class suturo:'Milk', as well as an owl:'NamedIndividual'.

Now that we know that a Milk object probably exists, even though the robot could not see it, we can ask where it usually is located so that the robot can look at the correct location for it. 
The following predicate can be used for that:

```prolog
storable_at(ObjectInstance, Location)

```
### <span style="color:red">Exercise</span>: Try to use this predicate in order to find out the location of a Milk object.

In [ ]:
# Add your code here

<details>

<summary> Solution </summary>

```python
knowrob.once("instance_of(Instance, suturo:'Milk'), storable_at(Instance, Location).")
```

**Result**

```python
{'Instance': 'http://www.ease-crc.org/ont/SUTURO.owl#Milk_REOCKXLU',
 'Location': 'http://www.ease-crc.org/ont/SOMA.owl#Refrigerator_OLGDBZRE'}
```

</details>

With this query we say that we are looking for an instance of an object of Class 'Milk', if such an object is found, we pass it on to the **storable_at** predicate, which returns us the Location where this object usually is located. 

Internally, the reasoning can be explained as follows:
- Objects of Class Milk have the quality of being Perishable.
- An Object of Class Refrigerator stores Objects which have the quality of being Perishable. 
- **Conclusion:** Since Milk is a Perishable Object and Refrigerators stores Perishable Objects, the Milk is likely located in the Refrigerator. 

You can try this out by using the **has_quality(Instance, Quality)** Predicate.

In [ ]:
# (Optional) Add your code here

<details>

<summary> Solution </summary>


```python
knowrob.all_solutions("instance_of(Instance, suturo:'Milk'), has_quality(Instance, Quality).")

Out[] {'Instance': 'http://www.ease-crc.org/ont/SUTURO.owl#Milk_EMRKTPVO',
      'Quality': 'http://www.ease-crc.org/ont/SUTURO.owl#Perishable'}

```

</details>

## <span style="color:red">Exercise</span>: Enable the robot to open the fridge

Now that we know that the Milk is usually found in the refrigerator, we need to enable the robot to open the fridge. For this, we need to find the fridge door, so that we can find the handle and extract the link name. We will guide you through the necessary queries step by step. Let's first access the refrigerator door.

<details>

<summary> <b> Hints </b> </summary>

**Note** you need to replace the parameters accordingly. This hint just tells you what kind of Argument the predicate expects.

```Prolog
instance_of(Instance, Class)
which_door(Instance, Instance)
```

You can assume, that the Angle of the joint is 0 for closed, and 1.57 for fully open.

</details>

In [ ]:
# Add your code here

<details>

<summary> Solution </summary>

```python
knowrob.once("instance_of(Refrigerator, soma:'Refrigerator'), which_door(Refrigerator, Door).")
```

</details>

Now that we have the door, we would like to know if it is opened or closed.

```Prolog
open(Door, Angle)
```

Assume that an angle of 0 means the door is closed, and an Angle of 1.57 means it is open. Feel free to try out both cases.
The angle can be read out from the joint_states topic like so (if you want): 

```python
joint_state_subscriber.get_joint_angle('refrigerator_door_top_left_joint')
```
You can read it out. If you don't want to, then assume that Angle = 0 is door closed, Angle = 1.57 is open.

In [ ]:
# Add your code here

<details>

<summary> Solution </summary>

```python
knowrob.once("instance_of(Refrigerator, soma:'Refrigerator'), which_door(Refrigerator,Door), open(Door, 1.57).")
```

</details>

Why do you think, that this is the result? Does it behave like you would have expected?

<details>

<summary> Solution </summary>

The expectation might have been, that you get True or False as a result, or as a variable binding. However, you get the variable bindings as a result only if prolog finds a match for them, which it does when the door is open and therefore you get all the variables bound. This means the condition is True. In the other case the result is False, and no bindings for the variables have been returned.

Now, we would like to get the handle of that door and obtain it's urdf link name.

<details>

<summary> Hint </summary>

```Prolog
which_handle(Instance, Instance)
has_urdf_name(Instance, LinkName)
```

</details>


In [ ]:
# Add your code here

<details>

<summary> <b> Solution </b> </summary>

```python
# query which finds out the handle name
query = "instance_of(Refrigerator, soma:'Refrigerator'), which_door(Refrigerator,Door), which_handle(Door,Handle), has_urdf_name(Handle, HandleLinkName)."
# Run the query on the knowledge base
result = knowrob.once(query)
handle_link= result.get("HandleLinkName")
print(handle_link)
```

We get the instance of the Refrigerator, then get it's door and handle, extract the link name of the handle. 

</details>

Now we have obtained all the necessary information that we need in order to open the door. It might feel tideous for now, since had to align several predecates. However, these can be also summarized within other predicates on the Prolog side, or as functions on the Python side. Whichever one would prefer. :)

### Step 4: Plan ahead for the next steps
Now that we know where the milk is located, we can plan the next steps. The robot needs to open the fridge, detect the milk, and grasp it. Let's start by moving the robot to the fridge door. Remember to use the NavigateAction to move the robot to the fridge door. The pose could be: [1.3, 2.5, 0], [0, 0, 1, 0]

<span style="color:red">Exercise</span>: Move the robot to the fridge door.


In [ ]:
 # Add your code here

<details>

<summary>Solution</summary>

```python
nav_pose = Pose([1.3, 2.5, 0], [0, 0, 1, 0])


with simulated_robot:
   NavigateAction(target_locations=[nav_pose]).resolve().perform()  
#If you see the robot moving you can continue with the next cell
```
</details>

The issue now is that the robot might not be able to open the fridge door from its current position. This is because the robot needs to be positioned directly in front of the fridge door and requires sufficient space to open it. Therefore, the robot needs to move to a better position. Manually adjusting its location for every object in the environment would be tedious. Instead, we can leverage the knowledge base to **query** the location of the fridge door handle and use **costmaps** to determine an optimal position for opening the door.





### Step 5: Understanding Costmaps


Costmaps are a way for robots to assess their surroundings by assigning numerical values (or "costs") to different areas of the environment based on specific criteria. This helps the robot understand which areas are visible, or preferable for certain tasks. The higher the value of a pose, the more likely it is to be chosen by the robot. If we choose to publish the costmap and visualize it inside RViz, each pose's color and z-coordinate are also determined by the poses value. The color follows a linear viridis gradient, with higher values mapped to brighter colors. Similarly, the z-coordinate increases with the pose's value, positioning higher-value poses at greater z-heights.

#### Types of Costmaps

 **a. Visibility Costmap**
A visibility costmap determines which poses around a target position can observe the target. This is especially useful for robots with cameras that can change their height, allowing them to adjust their view to detect objects or obstacles more effectively.

**Example Scenario**: If a robot needs to monitor a specific object, it uses a visibility costmap to identify which positions it should move to in order to keep that object in sight.

**b. Occupancy Costmap**
An occupancy costmap marks which areas in the environment are free of obstacles and safe for the robot to navigate. It essentially maps out all the positions where the robot can move without colliding with objects. The parameter "distance_to_obstacle" is used to set the minimum distance between the robot and any obstacle.

**Example Scenario**: When planning a path, the robot uses this map to avoid bumping into furniture or walls.

**c. Semantic Costmap**
A semantic costmap marks an area over a link of an object, that allows to dynamically determine potential poses on the surface of the object. This is useful for tasks where the robot needs to interact with objects or surfaces in a specific way.

**Example Scenario**: If a robot is looking to plae an object, we can use the semantic costmap to find a suitable location dynamically. 

**d. Gaussian Costmap**
A Gaussian costmap assigns values based on the distance from a certain region, with the highest values at the center of that region. The region can be determined by a "distance" parameter, where "distance" denotes the distance of the peak from the center of the costmap. A distance of 0 creates a costmap with a single peak at the center of the costmap. This is useful for tasks where proximity to a specific location is important.

**Example Scenario**: The robot uses this map to decide how close it should be to a person or object to interact effectively.

**e. Directional Costmap**
A directional costmap allows to create costmap that covers exactly half of the area around an object in one direction that may be specified. This is useful for tasks where the robot needs to interact with objects or surfaces in a specific direction.

**Example Scenario**: If an object needs to be interacted with from the front, this costmap can help finding poses that are in front of the object, instead of behind it.

#### Multiplying Costmaps
Robots can multiply different costmaps to factor in multiple criteria simultaneously. For instance, combining visibility and occupancy costmaps allows the robot to find a spot that is both visible and free of obstacles, since poses that cannot observe an object, or are too close to an obstacle, will have a cost of 0, leaving only the overlap of the two costmaps.

**Example Scenario**: If a robot is asked to monitor an area while avoiding collisions, it uses a combined costmap to determine an optimal position that satisfies both conditions.

#### Prioritizing Poses in Costmaps
Robots can prioritize poses of a primary costmap by using a secondary costmap. The overlap of the two costmaps will be high priority poses, while poses in the primary costmap that were not part of the overlap will be lower priority, but not completely ignored.

**Example Scenario**: If a robot needs to place an object on a table, it can use a primary costmap to find possible poses from which to place from, and a directional costmap to determine from which side of the table to approach. Should all prioritized poses fail for some reason, the low priority poses may still be tried out, as as a fallback.

By using costmaps, robots can make informed decisions about where to move or position themselves, enabling them to navigate and interact with their environment more intelligently.

You can find the example costmaps in the picture below:


![Costmap1](img/costmaps.001.jpeg)


## Step 6: Opening Action
Opening allows the robot to open a Container, the container is identified by an ObjectPart designator which describes the handle of the drawer that should be grasped. The OpeningAction needs to know which arm should be used to open the container. The ObjectPart would look like this:  

```python
ObjectPart(names=["refrigerator_door_top_handle"], part_of=apartment_desig)
```  

It takes the name of the handle as a string and the part_of designator of the apartment. This name is corresponding to the name of the handle in the URDF file. As you can tell naming all the parts in the URDF file is crucial for the robot to be able to interact with them, but knowing them during coding is quite annoying and frustrating. But we can use our knowledge base to ask for specific parts or names of objects.  

**Question**: Do you know what type of joint the fridge door has?
<details>

<summary>Solution</summary>

It is a revolute joint.
</details>

With the following query, we can obtain the name of the handle which the robot has to manipulate in order to open the fridge.

In [ ]:
query = "instance_of(Refrigerator, soma:'Refrigerator'), which_door(Refrigerator,Door), which_handle(Door,Handle), has_urdf_name(Handle, HandleLinkName)."
knowrob_result = knowrob.once(query)
handle_link_name = knowrob_result.get("HandleLinkName")
print(handle_link_name)

Another side note, how does the robot now actual knows that it has to open fridge, obviously we have to tell it. But how? We can use the knowledge base to ask for required actions. 

In [ ]:
# Add your code here

<details>

<summary> Hints </summary>

```prolog
instance_of(Instance, Class)
which_door(Instance, Instance)
which_action(Instance, Angle, Action)
```

</details>



<details>

<summary> Solution </summary>

```python
# check if we need to perform an opening or close action, depending on the angle of the door. 0 = Door is closed, 1.5 = Door is open.
# one can read out the joint state and pass it on as a parameter of course, since this is just a brief example we skip that.
query = "instance_of(Refrigerator, soma:'Refrigerator'), which_door(Refrigerator,Door), which_action(Door, 1.5, Action)." 
knowrob_result = knowrob.once(query)
print(knowrob_result.get("Action"))
```

The output of that query should roughly look like: 

```python
Action: 'CloseAction',
Door: 'http://www.ease-crc.org/ont/SOMA.owl#Door_MRIXTYLF',
Refrigerator: 'http://www.ease-crc.org/ont/SOMA.owl#Refrigerator_GMLTOIKJ'.
```

</details>

## <span style="color:red">Exercise</span>: Execute OpenAction
 Now write the code to open the fridge door, and then detect the milk.
Remember: Use the Costmaps to determine the optimal position for the robot to open the fridge door, query the knowledge base for the fridge door handle, and then open the fridge door. The costmap will help you determine the optimal position for the robot to open the fridge door, the method is called AccessingLocation. Please use the closed_location as the start_location inside the 'OpenAction' method.
```python
closed_location, opened_location = AccessingLocation(handle_desig=handle_designator.resolve(),
                                                         robot_desig=robot_desig.resolve()).resolve()
```

The `OpenAction` method allows a robot to open an object like a door or cabinet using its arm. Here's a quick guide on how to use it. 
#### Usage

To open an object, you need:

- An **object designator** that specifies what the robot should interact with (e.g., a handle).
- The **arm** that the robot will use for the action.
- The **start and goal locations** that define the positions before and after opening the object.

```python
OpenAction(
    object_designator_description=_, 
    arms=[closed_location.arms[0]], 
    start_goal_location=[_, opened_location]
).resolve().perform()
```
## Step 7: Writing the full plan

## <span style="color:red">Exercise</span>:  Write the plan

With all the queries that you have learned, you should be able to write the code to open the fridge door and detect the milk.

<details>

<summary>Hint 1</summary>

```python
# Python
with simulated_robot:
    start_pose = Pose([1.3, 2.7, 0], [0, 0, 1, 0])
    milk_target_pose = Pose([5.34, 3.55, 0.8])

    NavigateAction
    ParkArmsAction
    MoveTorsoAction

```

```prolog
% Prolog
instance_of(Instance, Class)
which_door(Instance, Instance)
which_handle(Instance, Instance)
has_urdf_name(Instane, UrdfLInkName)
which_action(Instance, Angle, Action)
```
</details>
<details>

<summary>Hint 2</summary>

```python

    query = "instance_of(Refrigerator, soma:'Refrigerator'), which_door(Refrigerator,Door), which_handle(Door,Handle), has_urdf_name(Handle, HandleLinkName)."
    knowrob_result = knowrob.once(query)
    handle_link_name = knowrob_result.get("HandleLinkName")
       
    handle_designator = ObjectPart(names=[handle_link_name], part_of=apartment_desig.resolve())
    closed_location, opened_location = AccessingLocation(handle_desig=handle_designator.resolve(),
                                                         robot_desig=robot_desig.resolve()).resolve()
    OpenAction(object_designator_description=handle_designator, arms=[closed_location.arms[0]],
               start_goal_location=[_, _]).resolve().perform()
    LookAtAction
    DetectAction
```
</details>

In [ ]:
#Add: your code here

<details>

<summary>Solution</summary>

```python

with simulated_robot:
    start_pose = Pose([1.3, 2.7, 0], [0, 0, 1, 0])
    milk_target_pose = Pose([5.34, 3.55, 0.8])

    NavigateAction([start_pose]).resolve().perform()
    ParkArmsAction([Arms.BOTH]).resolve().perform()
    MoveTorsoAction([TorsoState.HIGH]).resolve().perform()

    # get door handle link
    query = "instance_of(Refrigerator, soma:'Refrigerator'), which_door(Refrigerator,Door), which_handle(Door,Handle), has_urdf_name(Handle, HandleLinkName)."
    knowrob_result = knowrob.once(query)
    handle_link_name = knowrob_result.get("HandleLinkName")
    
    handle_designator = ObjectPart(names=[handle_link_name], part_of=apartment_desig.resolve())
    closed_location, opened_location = AccessingLocation(handle_desig=handle_designator.resolve(),
                                                         robot_desig=robot_desig.resolve()).resolve()
    # check if we need to perform an opening or close action, depending on the angle of the door. 0 = Door is closed, 1.5 = Door is open.
    query = "instance_of(Refrigerator, soma:'Refrigerator'), which_door(Refrigerator,Door), which_action(Door, 0, Action)." 
    knowrob_result = knowrob.once(query)
    if knowrob_result.get("Action") == 'OpenAction':
        OpenAction(object_designator_description=handle_designator, arms=[Arms.LEFT],
                    start_goal_location=[closed_location, opened_location]).resolve().perform()
                    
        LookAtAction(targets=[milk_desig.resolve().pose]).resolve().perform()
        object_designator = DetectAction(milk_desig).resolve().perform()
        print(object_designator)
```
</details>